# IncStrad – Microdati → DataFrame

Questo notebook trasforma i microdati ISTAT sugli **incidenti stradali** (formato fixed-width) in un DataFrame pandas.

**Struttura attesa delle cartelle:**
```
dati/
  INCSTRAD_<ANNO>_IT/
    MICRODATI/
      IncStrad_Microdati_Anno_<ANNO>.txt
    METADATI/
      IncStrad_Tracciato_Anno <ANNO>.html
      Classificazioni/
        IncStrad_Classificazione_Anno <ANNO>_var<N>.html
```

**Passaggi:**
1. Parsing del tracciato HTML → nome campo, lunghezza, tipo, link decodifica
2. Calcolo delle posizioni di inizio (somma cumulativa delle lunghezze)
3. Parsing delle tabelle di decodifica
4. Lettura dei file fixed-width con `pd.read_fwf`
5. Conversione tipi e applicazione decodifiche
6. Esportazione del DataFrame combinato

In [ ]:
!pip install -q beautifulsoup4 lxml pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Diagnostica: trova il percorso corretto su Google Drive

Se non sai il percorso esatto dei dati su Drive, esegui la cella qui sotto:
elencherà la struttura di Drive e **suggerirà il valore corretto di `BASE_DIR`**.

In [ ]:
# ================================================================
# DIAGNOSTICA — esegui questa cella per trovare BASE_DIR corretto
# poi aggiorna BASE_DIR nella cella CONFIGURAZIONE qui sotto
# ================================================================
import glob as _glob, os as _os

print('=== 1. Drive montato? ===')
drive_ok = _os.path.exists('/content/drive/MyDrive')
print('OK' if drive_ok else 'NO — decommenta drive.mount() nella cella sopra e riesegui')
print()

if drive_ok:
    print('=== 2. Primo livello di MyDrive ===')
    for item in sorted(_os.listdir('/content/drive/MyDrive'))[:30]:
        print(' ', item)
    print()

    print('=== 3. Ricerca file microdati .txt (può richiedere qualche secondo) ===')
    hits = _glob.glob('/content/drive/MyDrive/**/*.txt', recursive=True)
    microdati_hits = [h for h in hits if 'MICRODATI' in h]

    if microdati_hits:
        for h in microdati_hits[:10]:
            print(' ', h)
        sample = microdati_hits[0]
        # Risale 3 livelli: .../dati/INCSTRAD_ANNO_IT/MICRODATI/file.txt → dati/
        candidate = _os.path.normpath(
            _os.path.join(_os.path.dirname(sample), '..', '..', '..')
        )
        print(f'\n→ BASE_DIR suggerito: "{candidate}"')
        print('  Copia questo valore nella variabile BASE_DIR della cella CONFIGURAZIONE.')
    elif hits:
        print('  Trovati .txt ma nessuno in cartella MICRODATI:')
        for h in hits[:5]:
            print(' ', h)
    else:
        print('  Nessun file .txt trovato su Drive.')
        print('  Verifica di aver copiato la cartella dati/INCSTRAD_*/ su Google Drive.')

In [ ]:
# ================================================================
# CONFIGURAZIONE — modifica solo questi parametri
# ================================================================
import os

# Cartella che contiene le sottocartelle INCSTRAD_<ANNO>_IT/
BASE_DIR = '/content/drive/MyDrive/TesiMagistrale/dati'

# Encoding dei file di microdati e degli HTML (ISTAT usa latin-1)
DATA_ENCODING = 'latin-1'
HTML_ENCODING = 'latin-1'

# Se True, aggiunge colonne '<campo>_label' con le etichette decodificate
APPLY_DECODING = True

# Se True, aggiunge la colonna 'anno' ricavata dal path del file
ADD_YEAR_COLUMN = True

# Formato di esportazione: 'parquet' (consigliato), 'csv', 'pickle'
EXPORT_FORMAT = 'parquet'

# File di output (nella stessa cartella di BASE_DIR, un livello su)
OUTPUT_FILE = os.path.join(BASE_DIR, '..', f'incstrad_all.{EXPORT_FORMAT}')
OUTPUT_FILE = os.path.normpath(OUTPUT_FILE)

print(f'BASE_DIR   : {BASE_DIR}')
print(f'Esiste?    : {os.path.exists(BASE_DIR)}')
print(f'OUTPUT_FILE: {OUTPUT_FILE}')

In [ ]:
import os
import re
import glob
import warnings
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')
print('Librerie importate.')

## 1. Parsing del tracciato HTML

Il tracciato ha 13 colonne. Le posizioni di inizio **non sono nel file**: vengono calcolate come somma cumulativa delle lunghezze.

| Indice | Colonna         | Uso |
|--------|-----------------|-----|
| 1      | Lunghezza       | lunghezza del campo |
| 2      | Nome Campo      | nome colonna DataFrame |
| 3      | TipoVariabile   | link `Categorica` → tabella di decodifica |
| 8      | Formato Campo   | `A`=stringa, `N`=numerico |
| 9      | Num. decimali   | numero decimali (per i float) |
| 10     | Sep. decimali   | separatore decimale |

In [ ]:
def parse_tracciato(html_path, encoding=HTML_ENCODING):
    """
    Legge il tracciato HTML e restituisce:
      - fields: lista di dict {nome, start, length, formato, n_dec, sep_dec, descrizione}
      - decode_links: dict {nome_campo: path_assoluto_html_decodifica}

    La posizione iniziale (start, 0-based) è calcolata come somma cumulativa
    delle lunghezze dei campi precedenti.
    """
    with open(html_path, encoding=encoding, errors='replace') as f:
        soup = BeautifulSoup(f, 'lxml')

    table = soup.find('table')
    if table is None:
        raise ValueError(f'Nessuna tabella trovata in {html_path}')

    rows = table.find_all('tr')
    # Riga 0: titolo (colspan=13)  → saltare
    # Riga 1: intestazione (bgcolor=orange, usa <td> non <th>) → saltare
    # Righe 2+: dati

    decode_dir = os.path.dirname(html_path)
    fields = []
    decode_links = {}
    cursor = 0  # posizione corrente (0-based)

    for row in rows[2:]:
        cells = row.find_all('td')
        if len(cells) < 9:
            continue

        length_raw = cells[1].get_text(strip=True)
        nome = cells[2].get_text(strip=True)
        formato = cells[8].get_text(strip=True).upper()  # A o N
        n_dec_raw = cells[9].get_text(strip=True) if len(cells) > 9 else ''
        sep_dec = cells[10].get_text(strip=True) if len(cells) > 10 else ''
        descrizione = cells[7].get_text(strip=True) if len(cells) > 7 else ''

        # Salta righe non valide (testa tabelle annidate, celle vuote)
        if not nome or not re.match(r'^\d+$', length_raw):
            continue

        length = int(length_raw)
        n_dec = int(n_dec_raw) if re.match(r'^\d+$', n_dec_raw) else 0

        field = {
            'nome': nome,
            'start': cursor,
            'length': length,
            'formato': formato,
            'n_dec': n_dec,
            'sep_dec': sep_dec,
            'descrizione': descrizione,
        }
        fields.append(field)

        # Cerca link di decodifica in cells[3] (TipoVariabile)
        link_tag = cells[3].find('a', href=True)
        if link_tag:
            href = link_tag['href'].replace('./', '', 1)  # rimuove ./ iniziale
            decode_links[nome] = os.path.join(decode_dir, href)

        cursor += length

    print(f'Campi trovati       : {len(fields)}')
    print(f'Larghezza record    : {cursor} caratteri')
    print(f'Campi con decodifica: {len(decode_links)}')
    return fields, decode_links

### Test: verifica il tracciato del 2010

In [ ]:
# Trova il primo tracciato disponibile per verifica
tracciati = sorted(glob.glob(
    os.path.join(BASE_DIR, '*', 'METADATI', 'IncStrad_Tracciato_Anno*.html')
))
print(f'Tracciati trovati: {len(tracciati)}')
for t in tracciati:
    print(' ', t)

if tracciati:
    sample_fields, sample_decode_links = parse_tracciato(tracciati[0])
    df_tracciato = pd.DataFrame(sample_fields)
    print()
    display(df_tracciato.head(10))
    # Verifica: start del campo 1 = 0, campo 2 = 2, campo 3 = 4, ...
    print('\nVerifica posizioni inizio:')
    print(df_tracciato[['nome', 'start', 'length', 'formato', 'n_dec']].to_string(index=False))

## 2. Parsing delle tabelle di decodifica

Ogni HTML di classificazione ha una tabella con intestazione `bgcolor=orange` (usa `<td>`, non `<th>`).
- Colonna 0: codice
- Colonna 1: etichetta

In [ ]:
def parse_decode_table(html_path, encoding=HTML_ENCODING):
    """
    Legge un HTML di classificazione e restituisce un dict {codice: etichetta}.
    La prima riga (bgcolor=orange) è l'intestazione e viene saltata.
    """
    if not os.path.exists(html_path):
        return {}

    with open(html_path, encoding=encoding, errors='replace') as f:
        soup = BeautifulSoup(f, 'lxml')

    table = soup.find('table')
    if table is None:
        return {}

    mapping = {}
    rows = table.find_all('tr')
    for row in rows[1:]:  # salta riga 0 (intestazione orange)
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        codice = cells[0].get_text(strip=True)
        etichetta = cells[1].get_text(strip=True)
        if codice and etichetta:
            mapping[codice] = etichetta

    return mapping

## 3. Lettura del file fixed-width e conversione tipi

In [ ]:
def read_microdata(filepath, fields, encoding=DATA_ENCODING):
    """
    Legge un file di microdati separato da TAB.
    I nomi delle colonne vengono assegnati nell'ordine del tracciato.
    """
    names = [f['nome'] for f in fields]

    df = pd.read_csv(
        filepath,
        sep='\t',
        names=names,
        header=None,
        encoding=encoding,
        dtype=str,
        on_bad_lines='warn',
    )
    return df


def convert_types(df, fields):
    """
    Converte i tipi delle colonne:
      - formato A  → stringa (strip spazi)
      - formato N, n_dec == 0 → intero (nullable Int64)
      - formato N, n_dec >  0 → float
    """
    for f in fields:
        col = f['nome']
        if col not in df.columns:
            continue

        if f['formato'] == 'N':
            if f['n_dec'] > 0:
                sep = f['sep_dec'] if f['sep_dec'] else '.'
                if sep != '.':
                    df[col] = df[col].str.replace(sep, '.', regex=False)
                df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')
            else:
                df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce').astype('Int64')
        else:
            df[col] = df[col].str.strip()

    return df


def apply_decodings(df, decode_tables):
    """
    Per ogni campo con tabella di decodifica aggiunge una colonna
    '<campo>_label' con l'etichetta testuale.
    La colonna originale (con il codice) viene mantenuta.
    """
    for campo, mapping in decode_tables.items():
        if campo in df.columns:
            df[campo + '_label'] = df[campo].astype(str).str.strip().map(mapping)
    return df


def extract_year(path):
    """Estrae il primo anno a 4 cifre (1900-2099) dal path."""
    m = re.search(r'((?:19|20)\d{2})', str(path))
    return int(m.group(1)) if m else None

## 4. Elaborazione di tutti gli anni

In [ ]:
# Trova tutti i file di microdati
data_files = sorted(glob.glob(
    os.path.join(BASE_DIR, '*', 'MICRODATI', '*.txt')
))
print(f'File di microdati trovati: {len(data_files)}')
for p in data_files:
    print(' ', p)

In [ ]:
all_dfs = []

for filepath in data_files:
    anno = extract_year(filepath)
    print(f'\n[{anno}] {os.path.basename(filepath)}')

    # Trova il tracciato nella cartella METADATI dello stesso anno.
    # Cerca file con "incstrad" E "tracciato" nel nome (case-insensitive).
    # Questo copre tutte le varianti di naming:
    #   IncStrad_Tracciato_Anno 2010.html
    #   INCSTRAD_Tracciato_Anno 2013.html
    #   INCSTRAD_Tracciato_2017.html
    metadati_dir = os.path.normpath(
        os.path.join(os.path.dirname(filepath), '..', 'METADATI')
    )
    try:
        tracciati = [
            os.path.join(metadati_dir, f)
            for f in os.listdir(metadati_dir)
            if 'incstrad' in f.lower()
            and 'tracciato' in f.lower()
            and f.lower().endswith('.html')
        ]
    except FileNotFoundError:
        tracciati = []

    if not tracciati:
        print(f'  ATTENZIONE: nessun tracciato trovato in {metadati_dir}')
        continue

    tracciato_path = tracciati[0]
    print(f'  Tracciato: {os.path.basename(tracciato_path)}')

    try:
        # 1. Parsing tracciato
        fields, decode_links = parse_tracciato(tracciato_path)

        # 2. Caricamento tabelle di decodifica
        decode_tables = {}
        if APPLY_DECODING:
            for campo, html_path in decode_links.items():
                tbl = parse_decode_table(html_path)
                if tbl:
                    decode_tables[campo] = tbl
            print(f'  Tabelle di decodifica caricate: {len(decode_tables)}')

        # 3. Lettura file fixed-width
        df = read_microdata(filepath, fields)

        # 4. Conversione tipi
        df = convert_types(df, fields)

        # 5. Decodifiche
        if APPLY_DECODING:
            df = apply_decodings(df, decode_tables)

        # 6. Colonne anno e source
        # Il tracciato ha già un campo 'anno' (ultime 2 cifre):
        # lo rinominiamo per evitare conflitti con l'anno a 4 cifre.
        if ADD_YEAR_COLUMN:
            if 'anno' in df.columns:
                df = df.rename(columns={'anno': 'anno_ult2'})
            df.insert(0, 'anno', anno)
        df['_source_file'] = os.path.basename(filepath)

        all_dfs.append(df)
        print(f'  Righe: {len(df):,}  |  Colonne: {len(df.columns)}')

    except Exception as e:
        import traceback
        print(f'  ERRORE: {e}')
        traceback.print_exc()

print('\n' + '='*60)
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f'DataFrame totale: {len(df_all):,} righe × {len(df_all.columns)} colonne')
else:
    print('Nessun file elaborato con successo.')

## 5. Ispezione del DataFrame

In [ ]:
print(df_all.dtypes.to_string())

In [ ]:
df_all.head(3)

In [ ]:
# Riepilogo per anno
if ADD_YEAR_COLUMN:
    display(
        df_all.groupby('anno').size().rename('righe').reset_index()
    )

In [ ]:
# Anteprima campi con decodifica (es. mese, provincia)
cols_example = ['anno', 'mese', 'mese_label', 'provincia', 'provincia_label']
cols_present = [c for c in cols_example if c in df_all.columns]
if cols_present:
    display(df_all[cols_present].head(10))

## 6. Esportazione

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_FILE) or '.', exist_ok=True)

if EXPORT_FORMAT == 'csv':
    df_all.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
elif EXPORT_FORMAT == 'parquet':
    df_all.to_parquet(OUTPUT_FILE, index=False)
elif EXPORT_FORMAT == 'pickle':
    df_all.to_pickle(OUTPUT_FILE)
else:
    raise ValueError(f'Formato non supportato: {EXPORT_FORMAT}')

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'Salvato: {OUTPUT_FILE}  ({size_mb:.1f} MB)')

In [ ]:
# Opzionale: scarica il file nel browser
# from google.colab import files
# files.download(OUTPUT_FILE)

---

## 7. Analisi: incidenti in ambito urbano con utenti deboli o scontri frontali

Filtri applicati:
- **`localizzazione_incidente`** in [0, 1, 2, 3]
- **`natura_incidente`** == `05` **oppure** almeno uno dei veicoli coinvolti (a/b/c) ha **`tipo_veicolo`** tra 14 e 17 (velocipede, motociclo, altri utenti deboli)

Aggregazione per **anno × provincia × comune**.

In [ ]:
# ── Filtro 1: localizzazione_incidente in [0, 1, 2, 3] ──────────────────────
loc_mask = pd.to_numeric(df_all['localizzazione_incidente'], errors='coerce').between(0, 3)

# ── Filtro 2a: natura_incidente == '05' ──────────────────────────────────────
nat_mask = df_all['natura_incidente'].astype(str).str.strip() == '05'

# ── Filtro 2b: almeno uno dei veicoli (a/b/c) ha tipo in [14..17] ────────────
veic_cols = ['tipo_veicolo_a', 'tipo_veicoli__b_', 'tipo_veicolo__c_']
veic_cols_present = [c for c in veic_cols if c in df_all.columns]

veic_num = df_all[veic_cols_present].apply(pd.to_numeric, errors='coerce')
veic_mask = veic_num.apply(lambda s: s.between(14, 17)).any(axis=1)

# ── Maschera combinata ───────────────────────────────────────────────────────
mask = loc_mask & (nat_mask | veic_mask)

df_filtered = df_all[mask].copy()
print(f'Righe selezionate: {len(df_filtered):,} su {len(df_all):,} totali')

In [ ]:
# ── Aggregazione per anno × provincia × comune ───────────────────────────────
df_analisi = (
    df_filtered
    .groupby(['anno', 'provincia', 'comune'], dropna=False)
    .agg(
        n_incidenti      = ('morti_entro_24_ore',  'count'),
        morti_24h        = ('morti_entro_24_ore',  'sum'),
        morti_30g        = ('morti_entro_30_giorni','sum'),
        feriti           = ('feriti',               'sum'),
    )
    .reset_index()
    .sort_values(['anno', 'provincia', 'comune'])
)

print(f'Righe nel risultato: {len(df_analisi):,}')
display(df_analisi.head(20))

### Verifica: `morti_entro_30_giorni` include o esclude `morti_entro_24_ore`?

- Se esiste almeno un record con `morti_entro_24_ore > morti_entro_30_giorni` → i due campi sono **indipendenti** (30gg = solo decessi tra 24h e 30gg)
- Se non esiste mai tale condizione → `morti_entro_30_giorni` è probabilmente **cumulativo** (include anche i morti nelle prime 24h)

In [ ]:
m24 = pd.to_numeric(df_all['morti_entro_24_ore'],   errors='coerce')
m30 = pd.to_numeric(df_all['morti_entro_30_giorni'], errors='coerce')

# Righe con almeno un morto (esclude i casi banali 0 == 0)
has_deaths = (m24 > 0) | (m30 > 0)

n_24_gt_30 = ((m24 > m30) & has_deaths).sum()
n_24_eq_30 = ((m24 == m30) & has_deaths).sum()
n_30_gt_24 = ((m30 > m24) & has_deaths).sum()
n_both_zero = (~has_deaths).sum()

print(f'Record con almeno un morto          : {has_deaths.sum():>10,}')
print(f'  morti_24h  > morti_30g            : {n_24_gt_30:>10,}')
print(f'  morti_24h == morti_30g  (entrambi>0): {n_24_eq_30:>10,}')
print(f'  morti_30g  > morti_24h            : {n_30_gt_24:>10,}')
print(f'Record con entrambi a zero          : {n_both_zero:>10,}')
print()

if n_24_gt_30 > 0:
    print('✔ Esiste almeno un record con morti_24h > morti_30g.')
    print('  → I due campi sono INDIPENDENTI:')
    print('    morti_entro_24_ore  = decessi nelle prime 24h')
    print('    morti_entro_30_giorni = decessi tra 24h e 30gg (escluse le 24h)')
    print()
    print('  Totale decessi = morti_entro_24_ore + morti_entro_30_giorni')
else:
    print('✘ Non esiste alcun record con morti_24h > morti_30g.')
    print('  → I due campi sono probabilmente CUMULATIVI:')
    print('    morti_entro_24_ore  = decessi nelle prime 24h')
    print('    morti_entro_30_giorni = decessi totali entro 30gg (include le 24h)')
    print()
    print('  Totale decessi = morti_entro_30_giorni (già comprensivo delle 24h)')

# Mostra qualche esempio con morti in entrambe le colonne
sample = df_all[has_deaths][['anno', 'provincia', 'comune',
                               'morti_entro_24_ore', 'morti_entro_30_giorni']].head(10)
print()
display(sample)